# Doctor Assistant — Classifier Evaluation Lab (Google Colab)

This notebook evaluates **one chest classifier at a time** on pinned, patient-identified NIH ChestX-ray14 data. It is for iterative model development—not the mixed system demo.

Workflow:

1. Run both models on the **validation** split.
2. Review per-label AUROC/AUPRC, calibration, subgroup results, and ranked errors.
3. Improve/retrain the custom model and rerun validation under a new `RUN_NAME`.
4. Select per-label thresholds from validation only.
5. Set `RUN_FINAL_TEST=True` once, after the model and thresholds are frozen.

> Do not tune against the final test output. A successful notebook run establishes a reproducible research benchmark, not clinical validity.

## 1 · Evaluation configuration

Choose a GPU runtime in **Runtime → Change runtime type**. Set `REPO_REF` to the branch or tag containing the evaluation scripts. Keep a distinct `RUN_NAME` for every model revision.

In [ ]:
from pathlib import Path

# Edit REPO_REF when needed; leave REPO_URL construction unchanged so a Markdown
# hyperlink cannot accidentally be passed to `git clone`.
REPO_SLUG = 'Hamza09Hamza/doctor_assistant'
REPO_URL = f'https://github.com/{REPO_SLUG}.git'
REPO_REF = 'classifier-evaluation-colab'
REPO_DIR = Path('/content/doctor_assistant')

DRIVE_ROOT = Path('/content/drive/MyDrive/doctor_assistant')
CUSTOM_CHECKPOINT = DRIVE_ROOT / 'checkpoints/chest_xray/best.pt'
RUN_NAME = 'chest_round_01'        # change for every checkpoint/model revision

RUN_XRV_BASELINE = True
RUN_CUSTOM_MODEL = False
RUN_THRESHOLD_SELECTION = False
RUN_FINAL_TEST = False             # turn on only after model + thresholds are frozen

VALID_IMAGES = 300
TEST_IMAGES = 2000
SEED = 42
BOOTSTRAP_SAMPLES = 100
SUBGROUP_MIN_IMAGES = 50

# Must match the architecture used to create CUSTOM_CHECKPOINT.
CUSTOM_BACKBONE = 'timm:densenet121'
CUSTOM_IMAGE_SIZE = 320

# Research operating target. Change deliberately and record the rationale.
SENSITIVITY_TARGET = 0.85
SPECIFICITY_FLOOR = 0.60
MIN_VALIDATION_POSITIVES = 20
MIN_VALIDATION_NEGATIVES = 20

CACHE_DIR = Path('/content/doctor_assistant_eval_cache')
ARTIFACT_DIR = DRIVE_ROOT / 'evaluations' / RUN_NAME

print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Custom checkpoint: {CUSTOM_CHECKPOINT}')

## 2 · Mount Drive, pull the exact code revision, and install dependencies

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        'This notebook needs a Google Colab kernel. In VS Code, connect the notebook '
        'to a Colab runtime and rerun this cell.'
    ) from exc

drive.mount('/content/drive')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import os, shutil, subprocess, sys

def run_git(*arguments, cwd=None):
    command = ['git', *map(str, arguments)]
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode:
        detail = result.stderr.strip() or result.stdout.strip() or 'no error text returned'
        raise RuntimeError(f"Git command failed:\n  {' '.join(command)}\n\n{detail}")
    return result.stdout.strip()

if any(character in REPO_URL for character in '[]()'):
    raise ValueError(
        f'REPO_URL must be a plain Git URL, not a Markdown link: {REPO_URL!r}'
    )

# Fail early with a useful message when the branch has not been pushed yet.
remote_ref = run_git('ls-remote', '--heads', REPO_URL, REPO_REF)
if not remote_ref:
    raise RuntimeError(
        f'Remote branch {REPO_REF!r} was not found at {REPO_URL}. '\
        f'Push it from VS Code with: git push -u origin {REPO_REF}'
    )

# A failed clone may leave /content/doctor_assistant without a .git directory.
# It is an ephemeral Colab working copy, so remove only that exact incomplete path.
if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    print(f'Removing incomplete Colab clone: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_git('clone', '--branch', REPO_REF, '--single-branch', REPO_URL, REPO_DIR)
else:
    if run_git('-C', REPO_DIR, 'status', '--porcelain'):
        raise RuntimeError(
            f'{REPO_DIR} has local Colab changes. Restart the runtime or preserve '
            'those changes before refreshing the repository.'
        )
    run_git('-C', REPO_DIR, 'fetch', 'origin', REPO_REF)
    local_branches = run_git('-C', REPO_DIR, 'branch', '--list', REPO_REF)
    if local_branches:
        run_git('-C', REPO_DIR, 'checkout', REPO_REF)
    else:
        run_git('-C', REPO_DIR, 'checkout', '-b', REPO_REF, '--track', f'origin/{REPO_REF}')
    run_git('-C', REPO_DIR, 'merge', '--ff-only', f'origin/{REPO_REF}')

REPO_COMMIT = run_git('-C', REPO_DIR, 'rev-parse', 'HEAD')
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

required_script = REPO_DIR / 'scripts/benchmark_chest_classifier.py'
if not required_script.is_file():
    raise RuntimeError(
        f'{required_script} is missing. Set REPO_REF to the branch containing the '
        'classifier-evaluation work and rerun this cell.'
    )
print(f'Repository commit: {REPO_COMMIT}')

In [ ]:
%%capture
!python -m pip install -q \
    'monai>=1.3' 'nibabel>=5.2' 'timm>=0.9' \
    'scipy>=1.12' 'scikit-learn>=1.4' 'scikit-image>=0.22' \
    'pydantic>=2.6' 'Pillow>=10' 'pandas>=2' 'matplotlib>=3.8' \
    'datasets>=2.18' 'huggingface_hub>=0.23' torchxrayvision
print('Evaluation dependencies installed.')

## 3 · Preflight the GPU and custom checkpoint

The notebook refuses to evaluate random custom weights. If the checkpoint is absent, disable `RUN_CUSTOM_MODEL` or upload the real `best.pt` to the configured Drive path.

In [ ]:
import json, platform, torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}; PyTorch: {torch.__version__}; Python: {platform.python_version()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

if RUN_CUSTOM_MODEL:
    if not CUSTOM_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f'Real checkpoint not found: {CUSTOM_CHECKPOINT}. '
            'Upload best.pt or set RUN_CUSTOM_MODEL=False.'
        )
    checkpoint = torch.load(CUSTOM_CHECKPOINT, map_location='cpu', weights_only=False)
    if not checkpoint.get('class_names'):
        raise ValueError('Checkpoint has no class_names; label order cannot be verified safely.')
    print('Custom checkpoint preflight:')
    print(f"  epoch={checkpoint.get('epoch')}  best_metric={checkpoint.get('best_metric')}")
    print(f"  classes={checkpoint['class_names']}")

manifest = {
    'run_name': RUN_NAME,
    'repo_url': REPO_URL,
    'repo_ref': REPO_REF,
    'repo_commit': REPO_COMMIT,
    'custom_checkpoint': str(CUSTOM_CHECKPOINT) if RUN_CUSTOM_MODEL else None,
    'valid_images': VALID_IMAGES,
    'test_images': TEST_IMAGES,
    'seed': SEED,
}
(ARTIFACT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')

## 4 · Helpers and artifact paths

In [ ]:
import subprocess

XRV_VALID = ARTIFACT_DIR / 'xrv_validation.json'
CUSTOM_VALID = ARTIFACT_DIR / 'custom_validation.json'
THRESHOLDS = ARTIFACT_DIR / 'selected_thresholds.json'
FINAL_TEST = ARTIFACT_DIR / 'final_test.json'

def run_checked(arguments):
    print('$', ' '.join(str(value) for value in arguments))
    subprocess.run([str(value) for value in arguments], check=True)

def benchmark_command(model, split, count, output, threshold_artifact=None):
    command = [
        sys.executable, REPO_DIR / 'scripts/benchmark_chest_classifier.py',
        '--model', model, '--split', split, '--n', count,
        '--seed', SEED, '--device', DEVICE,
        '--cache-dir', CACHE_DIR,
        '--bootstrap-samples', BOOTSTRAP_SAMPLES,
        '--subgroup-min-images', SUBGROUP_MIN_IMAGES,
        '--output', output,
    ]
    if model == 'custom':
        command += [
            '--checkpoint', CUSTOM_CHECKPOINT,
            '--backbone', CUSTOM_BACKBONE,
            '--image-size', CUSTOM_IMAGE_SIZE,
        ]
    if threshold_artifact is not None:
        command += ['--threshold-artifact', threshold_artifact]
    return command

## 5 · Validation baseline — pretrained TorchXRayVision

This establishes the public pretrained baseline on the exact cohort used for the custom model. It does not train or modify the model.

In [ ]:
if RUN_XRV_BASELINE:
    run_checked(benchmark_command('xrv', 'valid', VALID_IMAGES, XRV_VALID))
else:
    print('XRV validation skipped.')

## 6 · Validation candidate — your custom checkpoint

The custom checkpoint is evaluated independently with the same seed, sample count, dataset revision, metrics, and error-analysis rules.

In [ ]:
if RUN_CUSTOM_MODEL:
    run_checked(benchmark_command('custom', 'valid', VALID_IMAGES, CUSTOM_VALID))
else:
    print('Custom validation skipped.')

## 7 · Compare scorecards and find the weaknesses

For rare findings, AUPRC is often more informative than AUROC. The operating metrics shown here still use the unvalidated global `0.5` threshold; use them to diagnose score behavior, not as a deployment claim.

In [ ]:
import pandas as pd
from IPython.display import display

def load_report(path):
    return json.loads(Path(path).read_text())

reports = {}
if XRV_VALID.is_file(): reports['xrv'] = load_report(XRV_VALID)
if CUSTOM_VALID.is_file(): reports['custom'] = load_report(CUSTOM_VALID)
if not reports:
    raise RuntimeError('No validation reports found; run sections 5 or 6 first.')

macro_rows = []
label_tables = {}
for model_name, report in reports.items():
    scorecard = report['scorecard']
    row = {'model': model_name, **scorecard['macro'], **scorecard['study_level']}
    ci = scorecard.get('confidence_intervals', {}).get('macro', {})
    row['auroc_95ci'] = (
        f"{ci['auroc']['low']:.3f}–{ci['auroc']['high']:.3f}"
        if ci.get('auroc', {}).get('low') is not None else 'n/a'
    )
    macro_rows.append(row)
    table = pd.DataFrame(scorecard['per_label']).T.reset_index(names='label')
    table.insert(0, 'model', model_name)
    label_tables[model_name] = table

print('Study-level and macro scorecard')
display(pd.DataFrame(macro_rows).set_index('model'))

for model_name, table in label_tables.items():
    print(f'Per-label weaknesses — {model_name}')
    display(table.sort_values(['auprc', 'auroc'], na_position='first'))

if {'xrv', 'custom'} <= reports.keys():
    columns = ['label', 'positives', 'prevalence', 'auroc', 'auprc', 'brier', 'ece']
    comparison = label_tables['xrv'][columns].merge(
        label_tables['custom'][columns], on=['label', 'positives', 'prevalence'],
        suffixes=('_xrv', '_custom'),
    )
    comparison['auroc_delta_custom_minus_xrv'] = comparison.auroc_custom - comparison.auroc_xrv
    comparison['auprc_delta_custom_minus_xrv'] = comparison.auprc_custom - comparison.auprc_xrv
    print('Paired cohort comparison (descriptive deltas; inspect confidence intervals)')
    display(comparison.sort_values('auprc_delta_custom_minus_xrv'))

## 8 · Subgroups and ranked error cases

Inspect view-position, gender, and age-group performance. Small groups are marked insufficient rather than assigned unstable metrics.

In [ ]:
ERROR_MODEL = 'custom' if 'custom' in reports else 'xrv'
ERROR_LABEL = 'Pneumothorax'   # change to a weak/important label from the table above

report = reports[ERROR_MODEL]
subgroup_rows = []
for field, groups in report['subgroups'].items():
    for group, values in groups.items():
        if 'status' in values:
            subgroup_rows.append({'field': field, 'group': group, **values})
        else:
            subgroup_rows.append({
                'field': field, 'group': group,
                'images': values['images'], 'patients': values['patients'],
                'macro_auroc': values['macro']['auroc'],
                'macro_auprc': values['macro']['auprc'],
                'normal_any_fp': values['study_level']['normal_any_false_positive'],
            })
display(pd.DataFrame(subgroup_rows))

print(f'Ranked {ERROR_MODEL} errors for {ERROR_LABEL}')
display(pd.DataFrame(report['ranked_errors'][ERROR_LABEL]['false_negatives']).assign(kind='FN'))
display(pd.DataFrame(report['ranked_errors'][ERROR_LABEL]['false_positives']).assign(kind='FP'))

## 9 · Select thresholds from validation only

Choose the candidate to freeze. Threshold export is refused when a label lacks enough validation examples or cannot meet the declared sensitivity/specificity constraints.

In [ ]:
SELECTED_MODEL = 'custom' if RUN_CUSTOM_MODEL else 'xrv'
selected_validation = CUSTOM_VALID if SELECTED_MODEL == 'custom' else XRV_VALID

if RUN_THRESHOLD_SELECTION:
    run_checked([
        sys.executable, REPO_DIR / 'scripts/select_chest_thresholds.py',
        '--benchmark', selected_validation,
        '--sensitivity-target', SENSITIVITY_TARGET,
        '--specificity-floor', SPECIFICITY_FLOOR,
        '--min-positives', MIN_VALIDATION_POSITIVES,
        '--min-negatives', MIN_VALIDATION_NEGATIVES,
        '--output', THRESHOLDS,
    ])
    threshold_report = load_report(THRESHOLDS)
    threshold_table = pd.DataFrame(threshold_report['selection']['per_label'])
    display(threshold_table)
    print('Exportable:', threshold_report['thresholds_complete'])
    print('Blocked labels:', threshold_report['blocked_labels'])
else:
    print('Threshold selection skipped.')

## 10 · Frozen held-out test — explicit opt-in

This cell will not run unless `RUN_FINAL_TEST=True` and every per-label threshold passed the validation support/constraint gate. After running it, do not change the model or thresholds and present the same test result as final.

In [ ]:
if not RUN_FINAL_TEST:
    print('Final held-out test is OFF. Continue improving on validation.')
else:
    if not THRESHOLDS.is_file():
        raise FileNotFoundError('Run threshold selection first.')
    threshold_report = load_report(THRESHOLDS)
    if not threshold_report['thresholds_complete']:
        raise RuntimeError(
            'Threshold artifact is incomplete. Increase validation support or revise '
            'the declared constraints before touching the test split.'
        )
    run_checked(benchmark_command(
        SELECTED_MODEL, 'test', TEST_IMAGES, FINAL_TEST,
        threshold_artifact=THRESHOLDS,
    ))
    final_report = load_report(FINAL_TEST)
    display(pd.DataFrame([{
        'model': SELECTED_MODEL,
        **final_report['scorecard']['macro'],
        **final_report['scorecard']['study_level'],
    }]).set_index('model'))
    print(f'Frozen final artifact: {FINAL_TEST}')

## What to do after each validation round

- Record the weakest labels by AUPRC, false negatives, and calibration—not only macro AUROC.
- Review whether the weakness is data scarcity, label noise, preprocessing, thresholding, or representation capacity.
- Make one controlled training/model change, save a new checkpoint, change `RUN_NAME`, and rerun the same validation cohort.
- Do not promote a model merely because one aggregate metric improved; check per-label regressions and subgroup gaps.
- Move to the next classifier only after the current classifier has a frozen checkpoint, reproducible configuration, complete threshold artifact, and untouched held-out result.